# Stage 2: data, trainers and pipeline checks

Each section restarts on its own and saves to Drive. Before each session: push from your laptop, then run section 0, which refuses to continue from code that isn't pushed.

| Section | Runtime | Colab units (estimate) |
|---|---|---|
| 0. Setup | Same as the section you're running | About 0 |
| A. Data and GPT-4o solutions (**done, don't rerun**) | CPU | About 0 |
| B. Mini-pilot | CPU for the merge, then G4 | About 14 |
| C. Memory test | G4, then L4 | About 5 |
| D. Smoke runs | G4 | About 11 |
| E. Lengths and drift | G4 | About 2 |
| F. Decision inputs | Any | 0 |

Stage 2's limit is 50 units. **Push the `prereg-draft` tag before section B**, which refuses to run without it. If a section runs well over its estimate, stop it and check with Claude.

## 0. Setup

In [ ]:
import os

if not os.path.isdir("/content/kl-reseach"):
    !git clone https://github.com/JuaniLlaberia/kl-reseach.git /content/kl-reseach
%cd /content/kl-reseach
!git pull --ff-only
!pip install -q -r requirements.txt
!pip uninstall -q -y torchaudio torchvision

Checks the pinned versions and the commit. If it says torch is on the wrong version, restart the session and rerun this cell.

In [ ]:
%cd /content/kl-reseach
import subprocess
from pathlib import Path

import torch

from utils.env import check_pins, check_pushed, read_pins

REPO = Path("/content/kl-reseach")

installed = check_pins(REPO / "requirements.txt")
torch_pin = read_pins(REPO / "requirements.txt")["torch"]
if torch.__version__.split("+")[0] != torch_pin:
    raise RuntimeError(f"torch {torch.__version__} is loaded. Restart the session.")

COMMIT = check_pushed(REPO)
print("Commit:", COMMIT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print(installed)

Files in `/content` are lost when the runtime is recycled, so everything is saved to Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/kl-reseach")

Runs the test suite (CPU, no network). Nothing below should run if this fails.

In [ ]:
tests = subprocess.run(["python", "-m", "pytest", "tests/", "-q", "-p", "no:cacheprovider"], capture_output=True, text=True, cwd=REPO)
print(tests.stdout[-1500:])
if tests.returncode != 0:
    raise RuntimeError("The tests failed. Don't run anything until they pass.")

## A. Data and GPT-4o solutions

**This section is finished: 1,381 of 1,800 rows solved. Don't rerun the 50-prompt test or the full run.** The solutions stay on Drive, where the sections below read them.

Use a **CPU runtime** for this section, as nothing here needs a GPU.

Add your OpenRouter key under Colab secrets as `OPENROUTER_API_KEY` and enable it for this notebook.

### Chemistry L-3 rows

Checks the row counts against the lab log, and that the split gives exactly the committed file.

In [ ]:
from collections import Counter

from src.data.chem import check_counts, load_chem_rows, load_split, select_rows, split_rows

SPLIT_PATH = REPO / "data" / "splits" / "chem_l3.json"

rows = load_chem_rows()
check_counts(rows)
split = load_split(SPLIT_PATH)
if split_rows(rows) != split:
    raise RuntimeError("The split doesn't reproduce the committed file.")

train_rows = select_rows(rows, split, "train")
heldout_rows = select_rows(rows, split, "heldout")
kl_rows = select_rows(rows, split, "kl")

print("Rows per type:", dict(Counter(row["type"] for row in rows)))
print("Rows per task:", dict(Counter(row["details"]["task"] for row in rows)))
print("Train / held-out / KL:", len(train_rows), len(heldout_rows), len(kl_rows))

### 50-prompt test

One first try each on the same 50 train rows as the first test, with the shorter wording. It should cost about $0.15-0.28, and it counts towards the $15 limit. Its attempts stay in the log, so the full run continues from them instead of paying twice.

The first test used another wording, so its log stays on Drive as `chem_l3_gpt4o_attempts.jsonl`. It's never reused, but what it cost counts towards the limit.

**Stop after this and check the output with Claude before the full run.**

In [ ]:
import json

from google.colab import userdata

from src.data.chem import build_prompt, stratified_sample
from src.data.solutions import generate_solutions, load_attempts, project_full_run, summarize_attempts

API_KEY = userdata.get("OPENROUTER_API_KEY")
BUDGET_USD = 15.0
TEST_MAX_TOKENS = 1024
SOLUTIONS_DIR = DRIVE_ROOT / "data" / "solutions"
FIRST_TEST_LOG = SOLUTIONS_DIR / "chem_l3_gpt4o_attempts.jsonl"
LOG_PATH = SOLUTIONS_DIR / "chem_l3_gpt4o_attempts_v2.jsonl"

if not FIRST_TEST_LOG.exists():
    raise FileNotFoundError(f"{FIRST_TEST_LOG} is missing, so its cost can't be counted towards the limit.")
first_test_attempts = load_attempts(FIRST_TEST_LOG)
PRIOR_SPEND_USD = sum(record["cost_usd"] for record in first_test_attempts)

test_rows = stratified_sample(train_rows, 50, seed=1)
result = generate_solutions(test_rows, split, LOG_PATH, API_KEY, BUDGET_USD, TEST_MAX_TOKENS, max_tries=1, prior_spend_usd=PRIOR_SPEND_USD)

test_ids = {row["row_id"] for row in test_rows}
test_attempts = [record for record in load_attempts(LOG_PATH) if record["row_id"] in test_ids and record["attempt"] == 1]
summary = summarize_attempts(test_rows, test_attempts)
first_summary = summarize_attempts(test_rows, first_test_attempts)
projection = project_full_run(train_rows, test_rows, test_attempts, BUDGET_USD - PRIOR_SPEND_USD)

print(result)
print(f"First test cost, counted towards the limit: ${PRIOR_SPEND_USD:.4f}")
for name in ("first_try_pass_rate",):
    print(name, "| first test:", first_summary["overall"][name], "| this test:", summary["overall"][name])
for name in ("cost_per_call_usd", "completion_tokens_mean", "completion_tokens_max", "truncated", "no_valid_answer"):
    print(name, "| first test:", first_summary[name], "| this test:", summary[name])
print(json.dumps(summary, indent=2))
print(json.dumps(projection, indent=2))

Read the responses, to check that the wording works and that nothing valid is being rejected. It prints up to 10 correct responses, to see how often the reasoning really supports the answer, plus all the filling rows.

In [ ]:
from src.data.chem import get_gold
from src.eval.extract import extract_answer

by_id = {row["row_id"]: row for row in test_rows}
shown = Counter()
for record in test_attempts:
    row = by_id[record["row_id"]]
    answer = extract_answer(record["text"], row["type"])
    kind = "valid answer, correct" if record["correct"] else ("no valid answer" if answer is None else "valid answer, wrong")
    shown[kind] += 1
    if shown[kind] <= (10 if record["correct"] else 3) or row["type"] == "filling":
        print("=" * 80)
        print(f"{kind} | {row['details']['task']} | gold: {get_gold(row)} | extracted: {answer} | {record['completion_tokens']} tokens")
        print(build_prompt(row))
        print("-" * 40)
        print(record["text"][-1200:])
print(shown)

### Full run

**Only run this after checking the test together, and after adding your OpenRouter credits.** Set `APPROVED_FULL_RUN` to `True`, and set `FULL_MAX_TOKENS` from the test's token counts.

If it stops or crashes, rerun the cell: it continues from the log without repeating any paid call. The limit covers this log and the first test.

If the prompt wording changes again, use a new log name and add this log's cost to `PRIOR_SPEND_USD`, as the code refuses to mix wordings.

In [ ]:
APPROVED_FULL_RUN = False
FULL_MAX_TOKENS = None

if not APPROVED_FULL_RUN or FULL_MAX_TOKENS is None:
    raise RuntimeError("Not approved yet.")

result = generate_solutions(train_rows, split, LOG_PATH, API_KEY, BUDGET_USD, FULL_MAX_TOKENS, prior_spend_usd=PRIOR_SPEND_USD)
print(result)

### Solutions and log entry

Saves one solution per solved row, plus the summary for the lab log. Afterwards, copy the summary from Drive to `data/solutions/` in the repo and commit it. The solutions and attempt logs stay on Drive until the terms on sharing GPT-4o text are checked.

Compare the total against your OpenRouter dashboard.

In [ ]:
from src.data.solutions import save_solutions

attempts = load_attempts(LOG_PATH)
n_solutions = save_solutions(train_rows, attempts, SOLUTIONS_DIR / "chem_l3_gpt4o.jsonl")
summary = summarize_attempts(train_rows, attempts)
summary["commit"] = COMMIT
(SOLUTIONS_DIR / "chem_l3_gpt4o_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Solved {n_solutions} of {len(train_rows)} ({n_solutions / len(train_rows):.1%})")
print(f"Total: ${summary['cost_usd'] + PRIOR_SPEND_USD:.2f}, with ${PRIOR_SPEND_USD:.2f} from the first test and {summary['calls']} calls in this log")
print(json.dumps(summary, indent=2))

## B. Mini-pilot

A pipeline check on your Reasoning SLM models: can retention and KL run on a base, an SFT and a GRPO checkpoint from start to finish? **It isn't evidence.** The cells only print whether the checks passed, and the notebook has no plot or fit of forgetting against KL. If you open the scores and something tempts you to change the metric, log that.

### One-off: rebuild the merged models

Drive only has the adapters. Run this **once, in its own CPU runtime**: section 0, then these two cells, then disconnect and delete the runtime, so peft is never installed where anything is measured. It skips models that already exist, and takes about 5 minutes.

The merges are in bf16, as the original `merge_adapter` did. GRPO trained on top of SFT, so its adapter goes into the merged SFT model.

In [ ]:
!pip install -q --no-deps peft==0.21.2

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-0.5B"
BASE_REVISION = "060db6499f32faf8b98477b0a26969ef7d8b9987"
SLM = Path("/content/drive/MyDrive/slm-post-training")

def merge_adapter_once(base_path: str, revision: str | None, adapter: Path, out_dir: Path) -> None:
    if (out_dir / "config.json").exists():
        print(f"{out_dir.name}: already exists, skipping")
        return
    if not adapter.exists():
        raise FileNotFoundError(adapter)

    model = AutoModelForCausalLM.from_pretrained(base_path, revision=revision, dtype=torch.bfloat16)
    before = {name: weight.clone() for name, weight in model.state_dict().items()}
    merged = PeftModel.from_pretrained(model, str(adapter)).merge_and_unload()
    changed = sum(not torch.equal(before[name], weight) for name, weight in merged.state_dict().items())
    if changed == 0:
        raise RuntimeError(f"{adapter.name} didn't change any weight.")

    merged.save_pretrained(out_dir)
    AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION).save_pretrained(out_dir)
    print(f"{out_dir.name}: {changed} of {len(before)} tensors changed")

merge_adapter_once(BASE_MODEL, BASE_REVISION, SLM / "adapters" / "sft-r16-step400", SLM / "models" / "sft-merged")
merge_adapter_once(str(SLM / "models" / "sft-merged"), None, SLM / "adapters" / "grpo_full-step400", SLM / "models" / "grpo_full-step400-merged")

### Retention

Use a **G4 runtime**. Stop here if the tag check fails. First a smoke test of a few minutes, and then the suite on all 3 models, about 24 minutes each. If it crashes, rerun the cell: benchmarks already saved are skipped.

In [ ]:
import os

from src.eval.minipilot import check_retention_complete
from src.eval.retention import run_retention
from utils.env import check_tag

check_tag(REPO, "prereg-draft")
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

BASE_MODEL = "Qwen/Qwen2.5-0.5B"
BASE_REVISION = "060db6499f32faf8b98477b0a26969ef7d8b9987"
SLM = Path("/content/drive/MyDrive/slm-post-training")
MINIPILOT = {
    "base": (BASE_MODEL, BASE_REVISION),
    "sft": (str(SLM / "models" / "sft-merged"), None),
    "grpo": (str(SLM / "models" / "grpo_full-step400-merged"), None),
}
MINIPILOT_DIR = DRIVE_ROOT / "results" / "minipilot_gsm8k"

for path, _ in list(MINIPILOT.values())[1:]:
    if not Path(path, "config.json").exists():
        raise FileNotFoundError(f"{path} is missing. Run the one-off merge first.")

smoke = run_retention(BASE_MODEL, str(DRIVE_ROOT / "smoke" / "minipilot_gsm8k"), "minipilot_gsm8k", "base", BASE_REVISION, limit=2)
check_retention_complete(smoke, limit=2)
print("Retention smoke test passed")

In [ ]:
for checkpoint, (path, revision) in MINIPILOT.items():
    results = run_retention(path, str(MINIPILOT_DIR), "minipilot_gsm8k", checkpoint, revision)
    check_retention_complete(results)
    print(f"{checkpoint}: every benchmark has a score")

### KL

Samples 500 GSM8K test prompts with the chat template, then measures every KL variant in fp32. These values aren't comparable to the earlier "KL about 0.13". It checks that KL(base ‖ base) is 0 on every variant, that nothing that should be positive is negative, and that every prompt is covered. Copy `results/minipilot_gsm8k/` from Drive into the repo afterwards and commit it.

In [ ]:
from transformers import AutoTokenizer

from src.eval.minipilot import check_kl_is_zero, check_kl_not_negative, check_kl_rows, load_gsm8k_prompts
from src.kl.measure import load_model, measure_kl, save_kl_results
from src.kl.sampling import build_prompt_ids, sample_responses

BATCH_SIZE = 16
KL_BATCH_SIZE = 8
MAX_NEW_TOKENS = 512
SEEDS = {"base": 7, "sft": 11, "grpo": 13}

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION)
prompts = build_prompt_ids(tokenizer, load_gsm8k_prompts())
base = load_model(BASE_MODEL, torch.float32, "cuda", BASE_REVISION)
base_path = MINIPILOT_DIR / "base_responses.jsonl"
sample_responses(base, tokenizer, prompts, base_path, SEEDS["base"], BATCH_SIZE, MAX_NEW_TOKENS, {"model": BASE_MODEL, "revision": BASE_REVISION})

zero, rows = measure_kl(base, base, base_path, base_path, KL_BATCH_SIZE)
check_kl_is_zero(zero)
check_kl_rows(rows, len(prompts))
print("KL(base || base) is 0 on every variant")

for checkpoint in ("sft", "grpo"):
    path = MINIPILOT[checkpoint][0]
    model = load_model(path, torch.float32, "cuda")
    responses_path = MINIPILOT_DIR / f"{checkpoint}_responses.jsonl"
    sample_responses(model, tokenizer, prompts, responses_path, SEEDS[checkpoint], BATCH_SIZE, MAX_NEW_TOKENS, {"model": path})
    scalars, rows = measure_kl(base, model, base_path, responses_path, KL_BATCH_SIZE)
    check_kl_not_negative(scalars)
    check_kl_rows(rows, len(prompts))
    save_kl_results(scalars, rows, MINIPILOT_DIR / f"{checkpoint}.json", torch.float32)
    print(f"{checkpoint}: {len(scalars)} KL variants saved, checks passed")
    del model
    torch.cuda.empty_cache()

## C. Memory test

Full fine-tuning with 8-bit AdamW, gradient checkpointing and bf16 autocast. **Run the 3 cells below on G4, then again on L4**; timings go in a separate folder for each GPU. Running out of memory is saved as a result. GRPO uses the paper's 64 × 8 at up to 512 new tokens, and gets only 5 steps, as each step generates 512 completions.

Nothing here is saved as a checkpoint.

In [ ]:
gpu_tests = subprocess.run(["python", "-m", "pytest", "tests/", "-q", "-m", "gpu", "-p", "no:cacheprovider"], capture_output=True, text=True, cwd=REPO)
print(gpu_tests.stdout[-1500:])
if gpu_tests.returncode != 0:
    raise RuntimeError("The GPU tests failed. Don't run anything until they pass.")

In [ ]:
from src.data.chem import check_counts, load_chem_rows, load_split, select_rows
from src.data.solutions import load_solved_rows
from src.train.run import gpu_tag, project_run, run_config

rows = load_chem_rows()
check_counts(rows)
split = load_split(REPO / "data" / "splits" / "chem_l3.json")
train_rows = select_rows(rows, split, "train")
solved_rows = load_solved_rows(train_rows, DRIVE_ROOT / "data" / "solutions" / "chem_l3_gpt4o.jsonl")
print(f"{len(solved_rows)} solved rows | GPU: {gpu_tag()}")

In [ ]:
MEMORY_RUNS = ["qwen15_chem_sft_memory_s0", "qwen15_chem_grpo_memory_s0", "qwen05_chem_sft_memory_s0", "qwen05_chem_grpo_memory_s0"]
MEMORY_DIR = DRIVE_ROOT / "results" / "memory" / gpu_tag()

for name in MEMORY_RUNS:
    summary = run_config(REPO / "configs" / f"{name}.yaml", solved_rows, MEMORY_DIR, DRIVE_ROOT / "checkpoints", use_wandb=False, save=False, time_limit_minutes=20)
    print(name, "|", summary["status"], "| seconds per step:", summary["seconds_per_step"], "| peak GB:", summary["peak_memory_gb"], "| full pass:", project_run(summary, len(solved_rows)))

## D. Smoke runs

Use **G4**. About 60 steps per trainer on chemistry, logged to W&B, with a smaller GRPO group to stay within budget. Add `WANDB_API_KEY` and `WANDB_PROJECT` under Colab secrets first. Run the data cell in section C first if you've restarted.

These settings are only pipeline checks, and aren't pre-registered choices. What to look for:
- **SFT:** the loss falls. GRPO's loss isn't a measure of progress, so it isn't checked.
- **GRPO:** rewards vary within groups. A share of groups with no variation close to 1 means the reward gives GRPO little to learn from.
- **Both:** the checkpoint reloads with identical fp32 weights, which `run_config` checks before saying it finished. About 6 GB each on Drive.

In [ ]:
import json
import os

from google.colab import userdata

for name in ("WANDB_API_KEY", "WANDB_PROJECT"):
    os.environ[name] = userdata.get(name)

SMOKE_DIR = DRIVE_ROOT / "results" / "smoke"
CHECKPOINTS = DRIVE_ROOT / "checkpoints"

for name, minutes in (("qwen15_chem_sft_smoke_s0", 15), ("qwen15_chem_grpo_smoke_s0", 60)):
    summary = run_config(REPO / "configs" / f"{name}.yaml", solved_rows, SMOKE_DIR, CHECKPOINTS, time_limit_minutes=minutes)
    print(name, "|", summary["status"], "| steps:", summary["steps_finished"], "| seconds per step:", summary["seconds_per_step"], "| peak GB:", summary["peak_memory_gb"], "| checkpoint GB:", summary["checkpoint_gb"])
    print(json.dumps(summary["log"], indent=2))

## E. Lengths and drift

Use **G4**, and run the data cell in section C first if you've restarted. Both use **train prompts**, so the KL prompts and held-out rows stay unseen.

- **Lengths** feed `max_new_tokens`: base responses at temperature 1.0 with a cap of 1024, and the length of the SFT targets.
- **Drift** repeats the bf16 test on real smoke checkpoints, as Stage 1 only had noise added to the weights. It prints how far bf16 is from fp32, not the KL values, which are saved to Drive.

In [ ]:
import json

from transformers import AutoTokenizer

from src.data.chem import stratified_sample
from src.eval.new_task import build_chem_prompts, length_stats
from src.kl.measure import load_model, measure_kl
from src.kl.sampling import load_responses, sample_responses
from src.train.common import load_config
from src.train.sft import IGNORE_INDEX, build_sft_rows

config = load_config(REPO / "configs" / "qwen15_chem_sft_smoke_s0.yaml")
MODEL_NAME, MODEL_REVISION = config["model"], config["revision"]
LENGTHS_DIR = DRIVE_ROOT / "results" / "lengths"
DRIFT_DIR = DRIVE_ROOT / "results" / "drift_smoke"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
base = load_model(MODEL_NAME, torch.float32, "cuda", MODEL_REVISION)
info = {"model": MODEL_NAME, "revision": MODEL_REVISION}

length_rows = stratified_sample(train_rows, 300, seed=2)
responses_path = LENGTHS_DIR / "base_responses.jsonl"
sample_responses(base, tokenizer, build_chem_prompts(tokenizer, length_rows), responses_path, 7, 16, 1024, info)
_, records = load_responses(responses_path)
lengths = {"base_responses": length_stats([len(record["response_ids"]) for record in records], [record["truncated"] for record in records])}

targets = build_sft_rows(tokenizer, solved_rows, tokenizer.eos_token_id, config["max_length"])
lengths["sft_targets"] = length_stats([sum(label != IGNORE_INDEX for label in target["labels"]) for target in targets], [False] * len(targets))
lengths["sft_full_sequences"] = length_stats([len(target["input_ids"]) for target in targets], [False] * len(targets))
(LENGTHS_DIR / "lengths.json").write_text(json.dumps(lengths, indent=2), encoding="utf-8")
print(json.dumps(lengths, indent=2))

In [ ]:
drift_prompts = build_chem_prompts(tokenizer, stratified_sample(train_rows, 100, seed=3))
base_path = DRIFT_DIR / "base.jsonl"
sample_responses(base, tokenizer, drift_prompts, base_path, 7, 16, 512, info)
bf16_base = load_model(MODEL_NAME, torch.bfloat16, "cuda", MODEL_REVISION)

drift = {}
for method, seed in (("sft", 11), ("grpo", 13)):
    path = DRIVE_ROOT / "checkpoints" / f"qwen15_chem_{method}_smoke_s0" / "final"
    fine_tuned = load_model(str(path), torch.float32, "cuda")
    responses_path = DRIFT_DIR / f"{method}.jsonl"
    sample_responses(fine_tuned, tokenizer, drift_prompts, responses_path, seed, 16, 512, {"model": str(path)})

    fp32, _ = measure_kl(base, fine_tuned, base_path, responses_path, 8)
    bf16, _ = measure_kl(bf16_base, fine_tuned.to(torch.bfloat16), base_path, responses_path, 8, allow_bf16=True)
    drift[method] = {name: {"fp32": fp32[name], "bf16": bf16[name]} for name in fp32}

    print(method)
    for name in fp32:
        print(f"  {name:40s} bf16 differs from fp32 by {abs(fp32[name] - bf16[name]) / abs(fp32[name]):.2%}")
    del fine_tuned
    torch.cuda.empty_cache()

(DRIFT_DIR / "drift.json").write_text(json.dumps(drift, indent=2), encoding="utf-8")

## F. Decision inputs

Reads what sections C to E saved, so any runtime works. It only shows timings, memory, storage, lengths and reward variation, which are the inputs Part 3 of `docs/PREREG.md` may use. Paste the output to Claude, then copy `results/memory/`, `results/smoke/`, `results/lengths/` and `results/drift_smoke/` from Drive into the repo and commit them.

Once you've read it, delete `checkpoints/` on Drive, as nothing later needs the smoke checkpoints.

In [ ]:
import json

from src.train.run import project_run

N_ROWS = 1381
summaries = [json.loads(path.read_text(encoding="utf-8")) for path in sorted((DRIVE_ROOT / "results").glob("*/**/summary.json"))]

print(f"{'run':32s} {'gpu':24s} {'status':14s} {'s/step':>7s} {'steps':>6s} {'hours':>6s} {'units':>6s} {'peak GB':>8s} {'ckpt GB':>8s} {'no var':>7s} {'cut':>6s}")
number = lambda value, digits=1: "-" if value is None else f"{value:.{digits}f}"
decision_inputs = []
for summary in summaries:
    projection = project_run(summary, N_ROWS)
    decision_inputs.append({**summary, "projection": projection})
    log = summary["log"]
    print(
        f"{summary['run']:32s} {summary['gpu'][:24]:24s} {summary['status'][:14]:14s} {number(summary['seconds_per_step']):>7s} {projection['steps_per_pass']:>6d} "
        f"{number(projection['hours']):>6s} {number(projection['units']):>6s} {number(summary['peak_memory_gb']['reserved']):>8s} {number(summary['checkpoint_gb']):>8s} "
        f"{number(log['share_of_groups_with_no_reward_variation'], 2):>7s} {number(log['truncated_share'], 2):>6s}"
    )

(DRIVE_ROOT / "results" / "decision_inputs.json").write_text(json.dumps(decision_inputs, indent=2), encoding="utf-8")
print(json.dumps(json.loads((DRIVE_ROOT / "results" / "lengths" / "lengths.json").read_text(encoding="utf-8")), indent=2))
print("Retention costs about 4 units per 0.5B checkpoint and 7 per 1.5B checkpoint on G4, from Stage 1.")